# Kadane (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/Kadane_cpp.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `Kadane`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/Kadane_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/Kadane_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `FibRec` (PascalCase).

## Kadane: un algoritmo iterativo migliore

**Kadane**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma massima di un sottoarray non vuoto $a[l..r]$. È la versione compatta del capitolo: $cur\leftarrow\max(a[i],\,cur+a[i])$, $best\leftarrow\max(best,cur)$. Il ciclo fa $n-1$ iterazioni: $\Theta(n)$ passi e $\Theta(1)$ memoria, contro le $n(n+1)/2$ coppie $(l,r)$ della forza bruta ($\Theta(n^2)$) e contro il divide et impera ricorsivo ($\Theta(n\log_2 n)$): un buon algoritmo iterativo può battere la ricorsione.

Sull'esempio delle variazioni di prezzo del capitolo 17 il risultato è $43$. Poi, su array pseudo-casuali con valori fra $-50$ e $50$ (stesso generatore in tutti i linguaggi, seme $n$), le iterazioni di `Kadane`, le coppie esaminate dalla forza bruta e la verifica che i due algoritmi trovino la stessa somma massima; infine i tempi misurati.

In [1]:
%%writefile kadane_compact.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Kadane(a, n): somma massima di un sottoarray non vuoto di a[0..n-1]
int Kadane(int *a, int n)
{
    int best = a[0];
    int cur = a[0];                 // somma massima di un sottoarray che termina in i
    int i = 1;
    while (i < n)
    {
        cur = max(a[i], cur + a[i]);
        best = max(best, cur);
        i = i + 1;
    }
    return best;
}

// Forza bruta: somma massima e numero di coppie (l, r) esaminate
int BruteForceBest(int *a, int n, long long &pairs)
{
    int best = a[0];
    pairs = 0;
    for (int l = 0; l < n; l++)
    {
        int s = 0;
        for (int r = l; r < n; r++)
        {
            s = s + a[r];
            pairs++;
            if (s > best) best = s;
        }
    }
    return best;
}

// Interi fra -50 e 50 (generatore congruenziale lineare)
vector<int> RandomValues(int n, unsigned long long seed = 12345)
{
    unsigned long long x = seed;
    vector<int> a(n);
    for (int i = 0; i < n; i++)
    {
        x = (x * 1103515245ULL + 12345ULL) % 2147483648ULL;
        a[i] = (int)(x % 101ULL) - 50;
    }
    return a;
}

double NowMs()
{
    return chrono::duration<double, milli>(chrono::steady_clock::now().time_since_epoch()).count();
}

#ifndef NOMAIN
int main()
{
    int a[] = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
    printf("a = %s\n", Fmt(a, 16).c_str());
    printf("Kadane(a, 16) = %d\n", Kadane(a, 16));
    int b[] = {-3, -1, -2, -5};
    printf("a = %s  Kadane(a, 4) = %d\n", Fmt(b, 4).c_str(), Kadane(b, 4));
    printf("\n");
    printf("%6s %7s %11s %11s %9s  stessa somma?\n", "n", "Kadane", "BruteForce", "iterations", "pairs");
    vector<array<double, 3>> times;
    for (int n : {1000, 2000, 4000})
    {
        vector<int> v = RandomValues(n, n);
        long long pairs;
        double t0 = NowMs(); int k = Kadane(v.data(), n); double t1 = NowMs() - t0;
        t0 = NowMs(); int bf = BruteForceBest(v.data(), n, pairs); double t2 = NowMs() - t0;
        printf("%6d %7d %11d %11d %9lld  %s\n", n, k, bf, n - 1, pairs, k == bf ? "OK" : "ERROR");
        times.push_back({(double)n, t1, t2});
    }
    printf("\n");
    printf("Tempi misurati (dipendono dalla macchina):\n");
    for (auto &t : times)
    {
        printf("n=%-6d Kadane %9.4f ms   BruteForce %10.3f ms\n", (int)t[0], t[1], t[2]);
    }
    return 0;
}
#endif

Writing kadane_compact.cpp


In [2]:
!g++ -std=c++17 -O2 -o kadane_compact kadane_compact.cpp && ./kadane_compact

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
Kadane(a, 16) = 43
a = [-3, -1, -2, -5]  Kadane(a, 4) = -1

     n  Kadane  BruteForce  iterations     pairs  stessa somma?
  1000     547         547         999    500500  OK
  2000     981         981        1999   2001000  OK
  4000    1955        1955        3999   8002000  OK

Tempi misurati (dipendono dalla macchina):
n=1000   Kadane    0.0013 ms   BruteForce      0.343 ms
n=2000   Kadane    0.0022 ms   BruteForce      1.350 ms
n=4000   Kadane    0.0045 ms   BruteForce      6.067 ms


**Prova tu.** Cambia $n$: le coppie della forza bruta crescono come $n^2$, le iterazioni di Kadane come $n$.

In [3]:
%%writefile prova_kadane_compact.cpp
#define NOMAIN
#include "kadane_compact.cpp"

int main()
{
    // Prova con n = 8000: le coppie della forza bruta sono n(n+1)/2 = 32 004 000, le iterazioni di Kadane 7999.
    int n = 8000;
    vector<int> v = RandomValues(n);
    long long pairs;
    int bf = BruteForceBest(v.data(), n, pairs);
    printf("%d (%d, %lld)\n", Kadane(v.data(), n), bf, pairs);
    return 0;
}

Writing prova_kadane_compact.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_kadane_compact prova_kadane_compact.cpp && ./prova_kadane_compact

1903 (1903, 32004000)


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).